# Phishing Email Classification with NLP

### Cybersecurity Threat Detection — Banking-AI

Phishing is a common way for attackers to steal bank credentials. They send emails that look like they're from a trusted source, asking the user to "verify their account" via a fake link.

In this notebook:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of cybersecurity and threat detection terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the NLP task in the context of cybersecurity and threat detection and how text features are extracted.
- Implement a text classification pipeline and evaluate accuracy on representative banking queries.
- Evaluate robustness to varied phrasing and identify failure modes relevant to customer-facing deployment.

**Estimated time:** 35–45 minutes

**Collection context:** DDoS detection, phishing classification, behavioural biometrics, and endpoint security in banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Humans are the weakest link in security. AI filters can catch 99% of phishing emails before they ever reach a user's inbox.

**Input data used:** Email subject lines and body text snippets.

**What we predict:** Email Class (Legitimate vs Phishing).

**ML method used:** Multinomial Naive Bayes (a classic, highly effective tool for text classification).

**Learning goal:** Learn how to handle text data and detect "malicious intent" in language.

**Primary stakeholders:** security operations analysts, incident responders, CISOs, and fraud prevention teams.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Dataset

We'll create examples of typical bank emails and typical phishing attacks.

In [ ]:
legit_emails = [
    "Your monthly bank statement is ready for viewing.",
    "Welcome to our new mobile banking app!",
    "Reminder: Your credit card payment is due in 3 days.",
    "Confirmation of your recent wire transfer to John Doe.",
    "Update to our Terms of Service - please review.",
    "Security Alert: A new device logged into your account.",
    "Happy Birthday from your local bank branch!",
    "Your mortgage application has been received."
] * 50

phishing_emails = [
    "URGENT: Your account has been suspended! Click here to verify.",
    "Action Required: Abnormal activity detected. Login immediately.",
    "You have won a $1000 gift card from the bank! Claim now.",
    "Final Notice: Unpaid tax balance found. Pay via this link.",
    "Security breach at our servers. Reset your password at this portal.",
    "Dear Customer, your debit card is blocked. Unlock it here.",
    "New message from the bank. Download the attachment to read.",
    "Invest in crypto now! 500% returns guaranteed by the bank."
] * 50

df = pd.DataFrame({
    'text': legit_emails + phishing_emails,
    'label': [0] * len(legit_emails) + [1] * len(phishing_emails)
})

df = df.sample(frac=1).reset_index(drop=True)
print(f"Dataset created with {len(df)} emails.")

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

What words appear most in Phishing vs Legitimate emails?

In [ ]:
from collections import Counter

phish_words = " ".join(df[df['label']==1]['text']).lower().split()
legit_words = " ".join(df[df['label']==0]['text']).lower().split()

print("Top Phishing Words:", Counter(phish_words).most_common(5))
print("Top Legit Words:", Counter(legit_words).most_common(5))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=RANDOM_SEED)

tfidf = TfidfVectorizer(stop_words='english')
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

print("Phishing classifier trained.")

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the most frequent class ---
from collections import Counter
target_col = df.columns[-1]
majority_class, majority_n = Counter(df[target_col]).most_common(1)[0]
print(f"Majority-class baseline: always predict '{majority_class}' -> accuracy {majority_n/len(df):.3f}")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Legit', 'Phish'], yticklabels=['Legit', 'Phish'])
plt.title('Phishing Detection Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Heatmap titled "Phishing Detection Confusion Matrix" with Predicted on the x-axis and Actual on the y-axis. The heatmap reveals correlation strength and direction between variables.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

In [ ]:
test_emails = [
    "Hey, your account is compromised! Verify NOW!",
    "Your monthly savings interest has been credited."
]
predictions = model.predict(tfidf.transform(test_emails))
probs = model.predict_proba(tfidf.transform(test_emails))

for email, pred, prob in zip(test_emails, predictions, probs):
    label = "PHISHING" if pred == 1 else "LEGITIMATE"
    print(f"Email: '{email}'")
    print(f"Result: {label} (Confidence: {prob[pred]:.2%})\n")

Let's test a brand new email.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: classify new queries ---
print("Operational demonstration — real-time intent classification:\n")
sample_queries = [
    "Can you show me my account balance?",
    "I need to transfer money to savings",
    "My card was stolen, please help",
    "What interest rates do you offer?",
    "I want to close my account",
]
for q in sample_queries:
    intent = model.predict([q])[0]
    print(f'  "{q}"  ->  {intent}')

print("\nIn production, predicted intents would route queries to the correct service channel.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end cybersecurity and threat detection workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Threat detection models must balance security effectiveness with employee and customer privacy.
- False positives in security alerts cause alert fatigue and reduce team effectiveness.
- Behavioural biometrics raise consent and surveillance concerns that require clear policies.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in cybersecurity and threat detection?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.